In [4]:
import wandb
wandb.login()

# Download an artifact from wandb
api = wandb.Api()
artifact = api.artifact("coactivelearning/llama_3_8b_ultrafeedback/k_2_ultrafeedback_llama_3_8b_bf16_dips:v0")

# Download the files to a local directory
artifact_dir = artifact.download("./ultrafeedback_old_dips")

wandb: Downloading large artifact k_2_ultrafeedback_llama_3_8b_bf16_dips:v0, 68.51MB. 6 files... 
wandb:   6 of 6 files downloaded.  
Done. 0:0:0.9


In [ ]:
import torch
from transformers import (
    AutoConfig,
    AutoModel,
    AutoModelForCausalLM,
    AutoModelForSequenceClassification,
    AutoTokenizer,
    GenerationConfig,
    PretrainedConfig,
    PreTrainedModel,
    get_scheduler,
    AutoTokenizer
)
from peft import get_peft_model, LoraConfig, PeftModel

base_model = AutoModelForCausalLM.from_pretrained(
    "meta-llama/Llama-3.1-8B-Instruct",
    torch_dtype="auto",
    device_map="auto"
)

config = LoraConfig.from_pretrained(artifact_dir)
policy = PeftModel.from_pretrained(base_model, artifact_dir)

policy.generation_config.eos_token_id = None  # disable `pad_token_id` and `eos_token_id` because we want to generate tokens without truncation / padding
policy.generation_config.pad_token_id = None 

Loading checkpoint shards: 100%|██████████| 4/4 [00:03<00:00,  1.26it/s]


In [7]:
merged_model = policy.merge_and_unload()
merged_model.save_pretrained("merged_model", safe_serialization=False)
del policy; del merged_model; del base_model

In [5]:
!cp ultrafeedback_old_dips/tokenizer.json merged_model/tokenizer.json
!cp ultrafeedback_old_dips/tokenizer_config.json merged_model/tokenizer_config.json

In [ ]:
from vllm import LLM, SamplingParams
query_length = 256
response_length = 1024
template_length = 64
llm = LLM(model="merged_model", 
          task="generate", 
          max_model_len = query_length + response_length + template_length + 1, 
          tensor_parallel_size = 4, 
          gpu_memory_utilization = 0.8)

/home/will/.conda/envs/dips/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
from datasets import load_dataset
ds = load_dataset("openbmb/UltraFeedback")

In [5]:
from transformers import AutoTokenizer
from tqdm import tqdm, trange

ultrafeedback_instructions = ds["train"]["instruction"]
tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.1-8B-Instruct")
tokenizer.add_special_tokens({"pad_token": "[PAD]"})
prompt_token_ids = [tokenizer.apply_chat_template([{"role": "user", "content": instruction}],
                                                 add_generation_prompt=True,
                                                  padding = "max_length",
                                                  max_length = query_length + template_length,
                                                  truncation = True) for instruction in tqdm(ultrafeedback_instructions)
                                                  if len(tokenizer(instruction).input_ids) <= query_length]

sampling_params = SamplingParams(temperature=0.7, top_p=0.95)
print(f"Sampling {len(prompt_token_ids)} instructions.")

100%|██████████| 63967/63967 [00:40<00:00, 1574.45it/s]

Sampling 52639 instructions.


In [4]:
outputs = llm.generate(prompt_token_ids=prompt_token_ids[:20], sampling_params=sampling_params)

Processed prompts: 100%|██████████| 20/20 [00:03<00:00,  5.12it/s, est. speed input: 5568.76 toks/s, output: 5.12 toks/s]


In [1]:
outputs[0].outputs[0].text

NameError: name 'outputs' is not defined